# Visual Test: Models on Real Demo Videos

Run the 4 trained backbones on actual demo video frames with real bay polygons.
Draws **green** (free) / **red** (occupied) overlays on each bay for manual verification.

In [ ]:
import sys, json, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
from torchvision.models import (
    resnet50, ResNet50_Weights,
    resnet101, ResNet101_Weights,
    efficientnet_b3, EfficientNet_B3_Weights,
    convnext_small, ConvNeXt_Small_Weights,
)
import matplotlib.pyplot as plt

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
IMG_SIZE = 128
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f"Device: {DEVICE}")

## 1. Load Trained Models

Uses the models trained in the quick test notebook. Re-trains quickly if not available.

In [ ]:
def build_resnet50():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 2)
    for name, param in model.named_parameters():
        if not any(name.startswith(l) for l in ["layer2", "layer3", "layer4", "fc"]):
            param.requires_grad_(False)
    return model

def build_resnet101():
    model = resnet101(weights=ResNet101_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 2)
    for name, param in model.named_parameters():
        if not any(name.startswith(l) for l in ["layer2", "layer3", "layer4", "fc"]):
            param.requires_grad_(False)
    return model

def build_efficientnet_b3():
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    for i, block in enumerate(model.features):
        if i < 5:
            for param in block.parameters():
                param.requires_grad_(False)
    return model

def build_convnext_small():
    model = convnext_small(weights=ConvNeXt_Small_Weights.DEFAULT)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    for i, stage in enumerate(model.features):
        if i < 4:
            for param in stage.parameters():
                param.requires_grad_(False)
    return model

MODELS = {
    "ResNet50": build_resnet50,
    "ResNet101": build_resnet101,
    "EfficientNet-B3": build_efficientnet_b3,
    "ConvNeXt-Small": build_convnext_small,
}

# Try loading saved weights, fall back to quick training
model_dir = PROJECT_ROOT / "backend" / "model"
trained_models = {}

for name, builder in MODELS.items():
    pt_name = f"occupancy_{name.lower().replace('-', '_')}_pklot.pt"
    pt_path = model_dir / pt_name
    model = builder()
    if pt_path.exists():
        ckpt = torch.load(pt_path, map_location="cpu", weights_only=True)
        model.load_state_dict(ckpt["state_dict"])
        print(f"{name}: loaded from {pt_path.name}")
    else:
        print(f"{name}: NO WEIGHTS FOUND, using ImageNet pretrained (results will be random for occupancy)")
    model = model.to(DEVICE).eval()
    trained_models[name] = model

print(f"\n{len(trained_models)} models loaded.")

## 2. Load Spatial Configs & Frames

In [ ]:
CONFIGS_DIR = PROJECT_ROOT / "backend" / "state" / "dev-blank" / "canonical" / "spatial-configs"
VIDEOS_DIR = PROJECT_ROOT / "demo" / "videos"

# Load the active version from the manifest
with open(CONFIGS_DIR / "manifest.json") as f:
    manifest = json.load(f)

active_version = manifest["activeVersion"]
active_file = CONFIGS_DIR / "versions" / f"{active_version:06d}.json"

with open(active_file) as f:
    global_cfg = json.load(f)

print(f"Active config: v{active_version} ({active_file.name})")
print(f"Total bays in global config: {len(global_cfg['bays'])}")

# Project bays per camera (same logic as project_config_to_camera)
cameras = {}
for bay in global_cfg["bays"]:
    cam = bay.get("cameraId")
    if not cam:
        continue
    cameras.setdefault(cam, {
        "bays": [],
        "frameWidth": global_cfg.get("frameWidth", 1280),
        "frameHeight": global_cfg.get("frameHeight", 720),
    })
    cameras[cam]["bays"].append(bay)

# Find frame directories for each camera
for cam_id in cameras:
    cam_video_dir = VIDEOS_DIR / cam_id
    if cam_video_dir.exists():
        subdirs = [d for d in cam_video_dir.iterdir() if d.is_dir()]
        if subdirs:
            frames = sorted(subdirs[0].glob("frame_*.jpg"))
            cameras[cam_id]["frames_dir"] = subdirs[0]
            cameras[cam_id]["num_frames"] = len(frames)
            bay_ids = [b["id"] for b in cameras[cam_id]["bays"]]
            print(f"  {cam_id}: {len(cameras[cam_id]['bays'])} bays {bay_ids}, {len(frames)} frames")
        else:
            print(f"  {cam_id}: {len(cameras[cam_id]['bays'])} bays, NO FRAMES DIR")
    else:
        print(f"  {cam_id}: {len(cameras[cam_id]['bays'])} bays, NO VIDEO DIR")

## 3. Predict & Visualize

For each camera, pick a few frames, run all models, overlay predictions.

In [ ]:
@torch.no_grad()
def predict_bays(model, frame_img, bays):
    """Run model on all bay crops from a frame. Returns list of (bay_id, occupied, confidence)."""
    w, h = frame_img.size
    crops = []
    valid_bays = []

    for bay in bays:
        poly = bay["imagePolygon"]
        if not poly:
            continue
        # Bounding box from polygon (normalized coords)
        xs = [p[0] for p in poly]
        ys = [p[1] for p in poly]
        x1, y1, x2, y2 = min(xs)*w, min(ys)*h, max(xs)*w, max(ys)*h
        # Clamp
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(w, int(x2)), min(h, int(y2))
        if x2 <= x1 or y2 <= y1:
            continue
        crop = frame_img.crop((x1, y1, x2, y2))
        crops.append(val_transform(crop))
        valid_bays.append(bay)

    if not crops:
        return []

    batch = torch.stack(crops).to(DEVICE)
    logits = model(batch)
    probs = logits.softmax(1)  # [N, 2] — col 0=free, col 1=occupied

    results = []
    for i, bay in enumerate(valid_bays):
        occ_prob = probs[i, 1].item()
        occupied = occ_prob >= 0.5
        conf = occ_prob if occupied else (1 - occ_prob)
        results.append({
            "bay_id": bay["id"],
            "label": bay.get("label", bay["id"]),
            "occupied": occupied,
            "confidence": conf,
            "polygon": bay["imagePolygon"],
        })
    return results


def draw_predictions(frame_img, predictions, title=""):
    """Draw colored polygon overlays on frame."""
    overlay = frame_img.copy()
    draw = ImageDraw.Draw(overlay, "RGBA")
    w, h = frame_img.size

    for pred in predictions:
        poly = pred["polygon"]
        pts = [(p[0]*w, p[1]*h) for p in poly]

        if pred["occupied"]:
            fill = (255, 0, 0, 80)    # red, semi-transparent
            outline = (255, 0, 0, 200)
        else:
            fill = (0, 200, 0, 80)    # green, semi-transparent
            outline = (0, 200, 0, 200)

        draw.polygon(pts, fill=fill, outline=outline)

        # Label
        cx = sum(p[0] for p in pts) / len(pts)
        cy = sum(p[1] for p in pts) / len(pts)
        label = f"{pred['bay_id']}\n{pred['confidence']:.0%}"
        draw.text((cx-10, cy-10), label, fill=(255, 255, 255, 230))

    return overlay


print("Prediction functions ready.")

In [ ]:
# Pick frames at different points in each video (start, middle, end)
FRAME_PICKS = [1, None, None]  # None = computed as middle and 3/4

for cam_id, cam_info in cameras.items():
    if "frames_dir" not in cam_info:
        continue

    frames_dir = cam_info["frames_dir"]
    n = cam_info["num_frames"]
    bays = cam_info["bays"]
    frame_indices = [1, n // 2, int(n * 0.75)]

    print(f"\n{'='*60}")
    print(f"Camera: {cam_id} ({len(bays)} bays, {n} frames)")
    print(f"{'='*60}")

    for frame_idx in frame_indices:
        frame_path = frames_dir / f"frame_{frame_idx:06d}.jpg"
        if not frame_path.exists():
            continue

        frame_img = Image.open(frame_path).convert("RGB")
        n_models = len(trained_models)

        fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5))
        if n_models == 1:
            axes = [axes]

        for ax, (model_name, model) in zip(axes, trained_models.items()):
            t0 = time.time()
            preds = predict_bays(model, frame_img, bays)
            elapsed_ms = (time.time() - t0) * 1000

            overlay = draw_predictions(frame_img, preds)
            n_occ = sum(1 for p in preds if p["occupied"])
            n_free = sum(1 for p in preds if not p["occupied"])

            ax.imshow(overlay)
            ax.set_title(f"{model_name}\n{n_occ} occ / {n_free} free | {elapsed_ms:.0f}ms", fontsize=11)
            ax.axis("off")

        plt.suptitle(f"{cam_id} — frame {frame_idx}/{n}", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

## 4. Model Agreement

Check how often the 4 models agree on each bay across multiple frames.

In [ ]:
# Sample 10 frames per camera, check inter-model agreement
N_SAMPLE_FRAMES = 10

for cam_id, cam_info in cameras.items():
    if "frames_dir" not in cam_info:
        continue

    frames_dir = cam_info["frames_dir"]
    n = cam_info["num_frames"]
    bays = cam_info["bays"]
    sample_indices = np.linspace(1, n, N_SAMPLE_FRAMES, dtype=int)

    print(f"\n{cam_id}: sampling {N_SAMPLE_FRAMES} frames")

    # Track per-bay predictions across models and frames
    # disagreements[bay_id] = count of frames where models disagree
    disagreements = {bay["id"]: 0 for bay in bays}
    total_frames = 0

    for frame_idx in sample_indices:
        frame_path = frames_dir / f"frame_{frame_idx:06d}.jpg"
        if not frame_path.exists():
            continue
        frame_img = Image.open(frame_path).convert("RGB")
        total_frames += 1

        # Get predictions from all models
        all_preds = {}
        for model_name, model in trained_models.items():
            preds = predict_bays(model, frame_img, bays)
            for p in preds:
                all_preds.setdefault(p["bay_id"], {})[model_name] = p["occupied"]

        # Check agreement per bay
        for bay_id, model_preds in all_preds.items():
            values = list(model_preds.values())
            if len(set(values)) > 1:  # disagreement
                disagreements[bay_id] += 1

    print(f"  {'Bay':8s} {'Disagree':>10s} {'Rate':>8s}")
    print(f"  {'-'*28}")
    for bay_id, count in sorted(disagreements.items(), key=lambda x: -x[1]):
        rate = count / total_frames if total_frames > 0 else 0
        marker = " <<<" if rate > 0.3 else ""
        print(f"  {bay_id:8s} {count:10d} {rate:7.0%}{marker}")

    total_disagree = sum(disagreements.values())
    total_possible = len(bays) * total_frames
    print(f"\n  Overall agreement: {1 - total_disagree/total_possible:.1%}")